In [2]:
import pandas as pd

df = pd.read_csv("../data/processed/text_processed.csv")

product_df = df.groupby("asin").agg({
    "title": "first",
    "clean_text": lambda x: " ".join(x),
    "overall": ["mean", "count"],
    "category_text": "first"
}).reset_index()

product_df.columns = [
    "asin", "title", "combined_text",
    "avg_rating", "rating_count", "category_text"
]

product_df["popularity"] = product_df["rating_count"]

# Save once
product_df.to_csv("../data/processed/product_df.csv", index=False)

print("Saved product_df:", product_df.shape)

Saved product_df: (30247, 7)


In [3]:
product_df.columns

Index(['asin', 'title', 'combined_text', 'avg_rating', 'rating_count',
       'category_text', 'popularity'],
      dtype='str')

In [4]:
product_df.head()

,asin,title,combined_text,avg_rating,rating_count,category_text,popularity
0,1118461304,NaN,clear lead innovation one thing book seemed ob...,4.600000,45,unknown,45
1,1906487049,NaN,hard find kim design experienced knitter like ...,4.666667,3,unknown,3
2,6040985461,NaN,look like original look like original keep min...,5.000000,1,unknown,1
3,7301113188,Tupperware Freezer Square Round Container Set ...,tupperware freezer square round container set ...,5.000000,1,unknown,1
4,7861850250,2 X Tupperware Pure &amp; Fresh Unique Covered...,x tupperware pure fresh unique covered cool cu...,3.000000,1,unknown,1


In [5]:
# ================================
# SAFE IN-PLACE FIX (NO BREAKAGE)
# ================================

import numpy as np

# 1. Fix title NaNs
product_df["title"] = product_df["title"].fillna("")

# 2. Backup original combined_text (for safety/debugging)
if "combined_text_original" not in product_df.columns:
    product_df["combined_text_original"] = product_df["combined_text"]

# 3. FIX combined_text IN-PLACE (this is key)
product_df["combined_text"] = (
    product_df["title"].astype(str) + " " + product_df["combined_text"].astype(str)
).str.strip()

# Clean extra spaces
product_df["combined_text"] = product_df["combined_text"].str.replace(r"\s+", " ", regex=True)

# 4. Fix popularity IN-PLACE (but keep backup)
if "popularity_original" not in product_df.columns:
    product_df["popularity_original"] = product_df["popularity"]

product_df["popularity"] = np.log1p(product_df["rating_count"])

# 5. Sanity checks
print("✅ In-place fixes applied\n")

print("Empty combined_text rows:",
      (product_df["combined_text"].str.len() == 0).sum())

print("\nSample:")
print(product_df[["title", "combined_text", "popularity"]].head())

✅ In-place fixes applied

Empty combined_text rows: 0

Sample:
                                               title  \
0                                                      
1                                                      
2                                                      
3  Tupperware Freezer Square Round Container Set ...   
4  2 X Tupperware Pure &amp; Fresh Unique Covered...   

                                       combined_text  popularity  
0  clear lead innovation one thing book seemed ob...    3.828641  
1  hard find kim design experienced knitter like ...    1.386294  
2  look like original look like original keep min...    0.693147  
3  Tupperware Freezer Square Round Container Set ...    0.693147  
4  2 X Tupperware Pure &amp; Fresh Unique Covered...    0.693147  
